# 01 — 强化学习入门

## 1 — 课程中的位置

本 Notebook 是强化学习系列的第一课，面向**已经具备基础 Python 编程和概率论知识**的学习者。

| 前一课 | 本课 | 下一课 |
|--------|------|--------|
| 无（入门） | RL 概览与核心概念 | 02 — 马尔可夫决策过程 (MDP) |

本课为整个课程奠定概念基础，后续所有算法（Q-learning、Policy Gradient、DQN、PPO 等）都建立在这些核心思想上。


## 2 — 学习目标

完成本 Notebook 后，你应当能：

1. **理解** RL 的基本问题：智能体在环境中学习做决策
2. **描述** Agent-Environment 交互循环
3. **定义** 关键概念：state、action、reward、return、episode、trajectory、policy
4. **实现** GridWorld 上的随机策略交互
5. **计算** 折扣累积回报 (Discounted Return)
6. **区分** model-based vs model-free、on-policy vs off-policy
7. **理解** 探索-利用困境并通过 Bandit 实验直观感受
8. **对比** RL 与监督学习、规划方法的不同


## 3 — 什么是强化学习？

### 直觉

强化学习 (Reinforcement Learning, RL) 是一种**通过试错学习做决策**的范式。

**示例 1：孩子学习走路**
- 孩子（智能体）尝试迈步（动作）
- 站住 → 得到鼓励（正奖励）；摔倒 → 感到疼痛（负奖励）
- 逐渐学会保持平衡的策略

**示例 2：AlphaGo 下围棋**
- AlphaGo（智能体）在棋盘上落子（动作）
- 赢棋 → +1 奖励；输棋 → -1 奖励
- 通过数百万次自我对弈学会最优策略

**示例 3：自动驾驶**
- 车辆（智能体）控制方向盘、油门、刹车（动作）
- 安全到达目的地 → 正奖励；偏离车道 → 负奖励
- 目标：学习一个安全的驾驶策略

### 核心问题

> **智能体 (Agent) 如何通过与环境 (Environment) 的交互，学习一个策略 (Policy)，使得累积奖励 (Cumulative Reward) 最大化？**

RL 的独特之处在于：
- **没有监督信号** — 智能体不知道哪个动作"正确"，只能通过延迟的奖励信号推测
- **动作影响后续状态** — 当前决策影响未来的可能性（非独立同分布数据）
- **需要平衡探索和利用** — 尝试新事物 vs 坚持已知的好选择


## 4 — Agent-Environment 交互循环

RL 的核心是 Agent 与 Environment 之间的持续交互：

```

                    ┌─────────────────────┐
                    │                     │
                    │    Environment      │
                    │                     │
                    │     State s_t       │
                    └──────────┬──────────┘
                               │
                    s_t ◄──────┴───────► r_t
                    │                      ▲
                    │                      │
                    ▼                      │
              ┌──────────┐                │
              │          │────────────────┘
              │  Agent   │    action a_t
              │          │
              └──────────┘
```

**交互流程（一个时间步 $t$）：**

1. 环境处于状态 $s_t$
2. 智能体观察到 $s_t$（或部分观察 $o_t$）
3. 智能体根据策略 $\pi(a_t|s_t)$ 选择动作 $a_t$
4. 环境执行动作，转移到 $s_{t+1}$，并产生奖励 $r_t$
5. 重复

整个过程形成一个**轨迹** (Trajectory):

$$ \tau = (s_0, a_0, r_0, s_1, a_1, r_1, \ldots, s_T) $$


## 5 — 核心概念

### 状态 (State) $s_t$

在时刻 $t$，环境的完整描述。状态的集合称为**状态空间** $\mathcal{S}$。

- **完全可观测**：智能体看到完整状态 $s_t$
- **部分可观测**：智能体只能看到观测 $o_t = \mathcal{O}(s_t)$

### 动作 (Action) $a_t$

智能体在时刻 $t$ 采取的行为。动作的集合称为**动作空间** $\mathcal{A}$。

- **离散动作空间**：$\mathcal{A} = \{a_1, a_2, \ldots, a_n\}$（如：上、下、左、右）
- **连续动作空间**：$\mathcal{A} \subseteq \mathbb{R}^n$（如：方向盘角度 $\in [-45^\circ, 45^\circ]$）

### 奖励 (Reward) $r_t$

标量反馈信号 $r_t \in \mathbb{R}$，告诉智能体上一动作的好坏。

- 正奖励：鼓励该动作
- 负奖励：惩罚该动作
- 奖励是**即时**的，但目标是最大化**长期**回报

### 回报 (Return) $G_t$

从时刻 $t$ 开始的累积折扣奖励：

$$ G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1} = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots $$

其中 $\gamma \in [0, 1]$ 是**折扣因子 (Discount Factor)**：

- $\gamma$ 接近 0：短视，只看近期奖励
- $\gamma$ 接近 1：远视，考虑长远收益

### 回合 (Episode) & 轨迹 (Trajectory)

- **Episode**：从初始状态到终止状态的完整交互序列（如：一盘棋、一局游戏）
- **Trajectory**：$\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \ldots)$

### 策略 (Policy) $\pi(a|s)$

策略是智能体的**行为准则**，定义在每个状态下选择各动作的概率：

$$ \pi(a|s) = \mathbb{P}[A_t = a \mid S_t = s] $$

- **确定性策略 (Deterministic Policy)**：$\mu(s) = a$，给定状态输出唯一动作
- **随机策略 (Stochastic Policy)**：$\pi(a|s) \in [0, 1]$，输出动作概率分布


In [1]:
# ============================================================
# Cell 6: 导入与全局设置
# ============================================================
import sys
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")  # 无头渲染，不弹出窗口
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 确保能找到 rl_course
sys.path.insert(0, "/workspace/data/vggt-omega/rl")
from rl_course.envs import GridWorld, MultiArmedBandit

# 图片输出目录
FIGURES_DIR = "/workspace/data/vggt-omega/rl/outputs/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# 全局 matplotlib 样式
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.figsize": (8, 5),
})

# 设置随机种子确保可复现
np.random.seed(42)

print("所有导入成功！")
print(f"NumPy 版本: {np.__version__}")
print(f"GridWorld 可用: ✓")
print(f"MultiArmedBandit 可用: ✓")


所有导入成功！
NumPy 版本: 2.4.6
GridWorld 可用: ✓
MultiArmedBandit 可用: ✓


## 6.1 — 创建 GridWorld 环境

我们使用自定义的 `GridWorld` 环境来演示 Agent-Environment 交互。

**GridWorld 设定：**
- 5×5 网格，从左上角 (0,0) 出发，目标在右下角 (4,4)
- 动作：0=上, 1=右, 2=下, 3=左
- 每步奖励 -1（鼓励尽快到达），到达目标 +10
- 障碍物为不可通过的格子


In [2]:
# ============================================================
# Cell 8: 创建并观察 GridWorld
# ============================================================

# 创建一个 5x5 的 GridWorld
env = GridWorld(
    width=5,
    height=5,
    start_pos=(0, 0),
    goal_pos=(4, 4),
    blocked_positions=[(1, 1), (2, 2), (3, 3)],  # 对角线障碍
    step_reward=-1.0,
    goal_reward=10.0,
    max_steps=50,
    seed=42,
)

print(f"状态空间大小: {env.n_states} ({env.width}×{env.height})")
print(f"动作空间大小: {env.n_actions} (0:↑, 1:→, 2:↓, 3:←)")
print(f"起始位置: {env.start_pos}")
print(f"目标位置: {env.goal_pos}")
print(f"障碍位置: {env.blocked_positions}")
print(f"每步奖励: {env.step_reward}")
print(f"目标奖励: {env.goal_reward}")
print(f"最大步数: {env.max_steps}")
print()

# 重置环境
obs = env.reset()
print("初始状态（渲染）：")
print(env.render("ansi"))
print(f"初始状态 ID: {obs}")


状态空间大小: 25 (5×5)
动作空间大小: 4 (0:↑, 1:→, 2:↓, 3:←)
起始位置: (0, 0)
目标位置: (4, 4)
障碍位置: [(1, 1), (2, 2), (3, 3)]
每步奖励: -1.0
目标奖励: 10.0
最大步数: 50

初始状态（渲染）：
A . . . .
. # . . .
. . # . .
. . . # .
. . . . G
初始状态 ID: 0


## 6.2 — 随机策略交互循环

下面我们在 GridWorld 上运行一个**随机策略 (Random Policy)** 的完整 episode。在每个状态，均匀随机选择一个可用动作。


In [3]:
# ============================================================
# Cell 10: 运行一个完整的随机策略 Episode
# ============================================================

def run_random_episode(env, render_steps=False):
    '''在 GridWorld 上运行一个随机策略 episode，返回轨迹信息。'''
    obs = env.reset()
    done = False
    trajectory = {
        "states": [obs],
        "actions": [],
        "rewards": [],
        "next_states": [],
        "dones": [],
    }

    step = 0
    while not done:
        # 随机策略：从合法动作中均匀采样
        action = np.random.choice(env.get_available_actions())
        next_obs, reward, done, info = env.step(action)

        trajectory["actions"].append(action)
        trajectory["rewards"].append(reward)
        trajectory["next_states"].append(next_obs)
        trajectory["dones"].append(done)

        if not done:
            trajectory["states"].append(next_obs)

        if render_steps:
            print(f"Step {step}: action={action} ({env.ACTION_NAMES[action]}), "
                  f"state={obs} -> {next_obs}, reward={reward:+.1f}, done={done}")
            if done:
                print("最终渲染：")
                print(env.render("ansi"))

        obs = next_obs
        step += 1

    trajectory["states"].append(next_obs)  # 终止状态
    return trajectory, step

# 设置种子确保可复现
np.random.seed(42)
env.seed = 42
env.rng = np.random.RandomState(42)

traj, steps = run_random_episode(env, render_steps=True)
print(f"\nEpisode 总步数: {steps}")
print(f"总奖励: {sum(traj['rewards']):+.1f}")


Step 0: action=1 (→), state=0 -> 1, reward=-1.0, done=False
Step 1: action=3 (←), state=1 -> 0, reward=-1.0, done=False
Step 2: action=1 (→), state=0 -> 1, reward=-1.0, done=False
Step 3: action=1 (→), state=1 -> 2, reward=-1.0, done=False
Step 4: action=3 (←), state=2 -> 1, reward=-1.0, done=False
Step 5: action=3 (←), state=1 -> 0, reward=-1.0, done=False
Step 6: action=1 (→), state=0 -> 1, reward=-1.0, done=False
Step 7: action=1 (→), state=1 -> 2, reward=-1.0, done=False
Step 8: action=3 (←), state=2 -> 1, reward=-1.0, done=False
Step 9: action=3 (←), state=1 -> 0, reward=-1.0, done=False
Step 10: action=1 (→), state=0 -> 1, reward=-1.0, done=False
Step 11: action=1 (→), state=1 -> 2, reward=-1.0, done=False
Step 12: action=3 (←), state=2 -> 1, reward=-1.0, done=False
Step 13: action=1 (→), state=1 -> 2, reward=-1.0, done=False
Step 14: action=1 (→), state=2 -> 3, reward=-1.0, done=False
Step 15: action=3 (←), state=3 -> 2, reward=-1.0, done=False
Step 16: action=2 (↓), state=2 -> 

In [4]:
# ============================================================
# Cell 11: 可视化 GridWorld 轨迹
# ============================================================

def visualize_gridworld_trajectory(env, trajectory, save_path=None):
    '''可视化 GridWorld 上的轨迹路径。'''
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, env.width)
    ax.set_ylim(0, env.height)
    ax.set_aspect("equal")
    ax.set_title("GridWorld 随机策略轨迹", fontsize=14)

    # 绘制网格背景
    for r in range(env.height):
        for c in range(env.width):
            pos = (r, c)
            color = "white"
            if pos == env.goal_pos:
                color = "#FFC107"   # 黄色：目标
            elif pos in env.blocked_positions:
                color = "#333333"   # 深灰：障碍
            elif pos == env.start_pos:
                color = "#E3F2FD"   # 浅蓝：起点
            rect = Rectangle(
                (c, env.height - 1 - r), 1, 1,
                facecolor=color, edgecolor="gray", linewidth=0.5,
            )
            ax.add_patch(rect)

    # 绘制轨迹路径（连接状态中心）
    states = trajectory["states"]
    positions = []
    for s in states:
        r = s // env.width
        c = s % env.width
        positions.append((c + 0.5, env.height - 1 - r + 0.5))

    positions_arr = np.array(positions)
    ax.plot(positions_arr[:, 0], positions_arr[:, 1], "b-o",
            markersize=8, linewidth=2, alpha=0.7, label="轨迹")

    # 标注起点和终点
    ax.scatter([positions[0][0]], [positions[0][1]], c="green", s=200,
               marker="s", zorder=5, label="起点 (A)")
    ax.scatter([positions[-1][0]], [positions[-1][1]], c="red", s=200,
               marker="*", zorder=5, label="终点")

    # 添加步骤编号
    for i, (x, y) in enumerate(positions):
        if i % 2 == 0 or i == len(positions) - 1:  # 每隔一步标注
            ax.annotate(str(i), (x - 0.15, y + 0.15), fontsize=9, color="blue")

    ax.set_xlabel("列")
    ax.set_ylabel("行")
    ax.legend(loc="upper right", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(env.width))
    ax.set_yticks(range(env.height))

    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=120)
        print(f"图片已保存: {save_path}")
    plt.close(fig)


# 运行多个 episode 并可视化第一个
np.random.seed(42)
env.rng = np.random.RandomState(42)
traj, steps = run_random_episode(env)

fig_path = os.path.join(FIGURES_DIR, "01_gridworld_trajectory.png")
visualize_gridworld_trajectory(env, traj, save_path=fig_path)
print(f"轨迹路径: {traj['states']}")
print(f"轨迹长度: {len(traj['states'])} 步")


图片已保存: /workspace/data/vggt-omega/rl/outputs/figures/01_gridworld_trajectory.png
轨迹路径: [0, 1, 0, 1, 2, 1, 0, 1, 2, 1, 0, 1, 2, 1, 2, 3, 2, 7, 2, 7, 8, 7, 8, 9, 14, 9, 4, 3, 8, 9, 4, 3, 4, 9, 8, 13, 8, 9, 8, 9, 14, 13, 14, 13, 14, 13, 14, 9, 8, 3, 2]
轨迹长度: 51 步


/tmp/ipykernel_3493536/3599480079.py:61: UserWarning: Glyph 34892 (\N{CJK UNIFIED IDEOGRAPH-884C}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3599480079.py:61: UserWarning: Glyph 38543 (\N{CJK UNIFIED IDEOGRAPH-968F}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3599480079.py:61: UserWarning: Glyph 26426 (\N{CJK UNIFIED IDEOGRAPH-673A}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3599480079.py:61: UserWarning: Glyph 31574 (\N{CJK UNIFIED IDEOGRAPH-7B56}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3599480079.py:61: UserWarning: Glyph 30053 (\N{CJK UNIFIED IDEOGRAPH-7565}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3599480079.py:61: UserWarning: Glyph 

## 6.3 — 计算折扣回报 (Discounted Return)

回报 (Return) 是从当前时刻开始所有折扣奖励的累积：

$$ G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1} $$

下面我们在 GridWorld 的轨迹上计算每个时间步的 $G_t$。


In [5]:
# ============================================================
# Cell 13: 计算折扣回报 (Discounted Return)
# ============================================================

def compute_discounted_returns(rewards, gamma=0.99):
    '''计算每个时间步的折扣回报 G_t。

    Args:
        rewards: 奖励列表 [r_0, r_1, ..., r_{T-1}]
        gamma: 折扣因子

    Returns:
        returns: 长度 T 的数组，returns[t] = G_t
    '''
    T = len(rewards)
    returns = np.zeros(T)
    G = 0.0
    # 从后向前递推：G_t = r_t + gamma * G_{t+1}
    for t in reversed(range(T)):
        G = rewards[t] + gamma * G
        returns[t] = G
    return returns


# 使用之前生成的轨迹
rewards = traj["rewards"]
returns = compute_discounted_returns(rewards, gamma=0.9)

print("步数 | 奖励 r_t | 回报 G_t (γ=0.9)")
print("-" * 35)
for t, (r, g) in enumerate(zip(rewards, returns)):
    print(f" {t:2d}  | {r:+6.1f}  | {g:+8.3f}")

print(f"\n{'='*35}")
print(f"总奖励 (sum): {sum(rewards):+.1f}")
print(f"折扣回报 G_0: {returns[0]:+.3f}")
print()

# 不同折扣因子的影响
print("不同 γ 下的 G_0 对比：")
for gamma in [0.0, 0.5, 0.9, 0.99, 1.0]:
    ret = compute_discounted_returns(rewards, gamma=gamma)
    print(f"  γ = {gamma:.2f}  →  G_0 = {ret[0]:+.3f}")


步数 | 奖励 r_t | 回报 G_t (γ=0.9)
-----------------------------------
  0  |   -1.0  |   -9.948
  1  |   -1.0  |   -9.943
  2  |   -1.0  |   -9.936
  3  |   -1.0  |   -9.929
  4  |   -1.0  |   -9.921
  5  |   -1.0  |   -9.913
  6  |   -1.0  |   -9.903
  7  |   -1.0  |   -9.892
  8  |   -1.0  |   -9.880
  9  |   -1.0  |   -9.867
 10  |   -1.0  |   -9.852
 11  |   -1.0  |   -9.836
 12  |   -1.0  |   -9.818
 13  |   -1.0  |   -9.797
 14  |   -1.0  |   -9.775
 15  |   -1.0  |   -9.750
 16  |   -1.0  |   -9.722
 17  |   -1.0  |   -9.691
 18  |   -1.0  |   -9.657
 19  |   -1.0  |   -9.618
 20  |   -1.0  |   -9.576
 21  |   -1.0  |   -9.529
 22  |   -1.0  |   -9.477
 23  |   -1.0  |   -9.419
 24  |   -1.0  |   -9.354
 25  |   -1.0  |   -9.282
 26  |   -1.0  |   -9.202
 27  |   -1.0  |   -9.114
 28  |   -1.0  |   -9.015
 29  |   -1.0  |   -8.906
 30  |   -1.0  |   -8.784
 31  |   -1.0  |   -8.649
 32  |   -1.0  |   -8.499
 33  |   -1.0  |   -8.332
 34  |   -1.0  |   -8.147
 35  |   -1.0  |   -7.941

In [6]:
# ============================================================
# Cell 14: 可视化不同折扣因子对回报的影响
# ============================================================

def plot_returns_comparison(rewards, gammas, save_path=None):
    '''绘制不同折扣因子下的逐步回报。'''
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 左图：不同 γ 的 G_t 曲线
    ax1 = axes[0]
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(gammas)))
    for gamma, color in zip(gammas, colors):
        returns = compute_discounted_returns(rewards, gamma=gamma)
        ax1.plot(range(len(returns)), returns, "o-", color=color,
                 linewidth=2, markersize=5, label=f"γ={gamma:.2f}")

    ax1.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax1.set_xlabel("时间步 t")
    ax1.set_ylabel("回报 G_t")
    ax1.set_title("不同折扣因子下的回报变化")
    ax1.legend(loc="best", fontsize=9)
    ax1.grid(True, alpha=0.3)

    # 右图：γ 与 G_0 的关系
    ax2 = axes[1]
    gamma_values = np.linspace(0, 1, 50)
    g0_values = [compute_discounted_returns(rewards, gamma=g)[0] for g in gamma_values]
    ax2.plot(gamma_values, g0_values, "b-", linewidth=2)
    ax2.axhline(y=sum(rewards), color="r", linestyle="--",
                label=f"sum(r) = {sum(rewards):+.1f}")
    ax2.set_xlabel("折扣因子 γ")
    ax2.set_ylabel("G₀")
    ax2.set_title("折扣因子 γ 对初始回报 G₀ 的影响")
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=120)
        print(f"图片已保存: {save_path}")
    plt.close(fig)


plot_returns_comparison(rewards, [0.0, 0.5, 0.9, 0.99, 1.0],
                        save_path=os.path.join(FIGURES_DIR, "01_discounted_returns.png"))


/tmp/ipykernel_3493536/4283177437.py:37: UserWarning: Glyph 26102 (\N{CJK UNIFIED IDEOGRAPH-65F6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4283177437.py:37: UserWarning: Glyph 38388 (\N{CJK UNIFIED IDEOGRAPH-95F4}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4283177437.py:37: UserWarning: Glyph 27493 (\N{CJK UNIFIED IDEOGRAPH-6B65}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4283177437.py:37: UserWarning: Glyph 22238 (\N{CJK UNIFIED IDEOGRAPH-56DE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4283177437.py:37: UserWarning: Glyph 25253 (\N{CJK UNIFIED IDEOGRAPH-62A5}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4283177437.py:37: UserWarning: Glyph 19981 (\N{CJK UNIFIED IDEOGRAPH-4E0D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4283177437.py:37: UserWarning: Glyph 21516 (\N{CJK UN

图片已保存: /workspace/data/vggt-omega/rl/outputs/figures/01_discounted_returns.png


/tmp/ipykernel_3493536/4283177437.py:39: UserWarning: Glyph 23545 (\N{CJK UNIFIED IDEOGRAPH-5BF9}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/4283177437.py:39: UserWarning: Glyph 21021 (\N{CJK UNIFIED IDEOGRAPH-521D}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/4283177437.py:39: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/4283177437.py:39: UserWarning: Glyph 24433 (\N{CJK UNIFIED IDEOGRAPH-5F71}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/4283177437.py:39: UserWarning: Glyph 21709 (\N{CJK UNIFIED IDEOGRAPH-54CD}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)


In [7]:
# ============================================================
# Cell 15: 多个 Episode 的随机策略表现统计
# ============================================================

def run_multiple_episodes(env, n_episodes=100, gamma=0.99):
    '''运行多个 episode，收集统计数据。'''
    episode_returns = []
    episode_lengths = []
    episode_rewards = []

    for ep in range(n_episodes):
        env.rng = np.random.RandomState(42 + ep)
        traj, steps = run_random_episode(env)
        ret = compute_discounted_returns(traj["rewards"], gamma=gamma)[0]
        episode_returns.append(ret)
        episode_lengths.append(steps)
        episode_rewards.append(sum(traj["rewards"]))

    return {
        "returns": np.array(episode_returns),
        "lengths": np.array(episode_lengths),
        "total_rewards": np.array(episode_rewards),
    }


stats = run_multiple_episodes(env, n_episodes=200, gamma=0.9)

print("=== 随机策略在 GridWorld 上的表现 ===")
print(f"Episode 数量: {len(stats['lengths'])}")
print(f"平均长度: {stats['lengths'].mean():.1f} 步 (min={stats['lengths'].min()}, max={stats['lengths'].max()})")
print(f"平均总奖励: {stats['total_rewards'].mean():+.2f}")
print(f"平均折扣回报 G₀: {stats['returns'].mean():+.3f}")
print(f"成功到达目标的概率: {(stats['lengths'] < env.max_steps).mean()*100:.1f}%")

# 绘制分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(stats["lengths"], bins=20, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(stats["lengths"].mean(), color="red", linestyle="--", label=f"平均={stats['lengths'].mean():.1f}")
axes[0].set_xlabel("Episode 长度")
axes[0].set_ylabel("频次")
axes[0].set_title("Episode 长度分布")
axes[0].legend(fontsize=9)

axes[1].hist(stats["total_rewards"], bins=20, color="coral", edgecolor="white", alpha=0.85)
axes[1].axvline(stats["total_rewards"].mean(), color="red", linestyle="--", label=f"平均={stats['total_rewards'].mean():+.2f}")
axes[1].set_xlabel("总奖励")
axes[1].set_ylabel("频次")
axes[1].set_title("总奖励分布")
axes[1].legend(fontsize=9)

axes[2].hist(stats["returns"], bins=20, color="forestgreen", edgecolor="white", alpha=0.85)
axes[2].axvline(stats["returns"].mean(), color="red", linestyle="--", label=f"平均={stats['returns'].mean():+.3f}")
axes[2].set_xlabel("折扣回报 G₀ (γ=0.9)")
axes[2].set_ylabel("频次")
axes[2].set_title("折扣回报分布")
axes[2].legend(fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "01_random_policy_stats.png"),
            bbox_inches="tight", dpi=120)
print(f"\n统计图已保存到 outputs/figures/")
plt.close(fig)


=== 随机策略在 GridWorld 上的表现 ===
Episode 数量: 200
平均长度: 39.6 步 (min=10, max=50)
平均总奖励: -34.48
平均折扣回报 G₀: -8.980
成功到达目标的概率: 45.5%


/tmp/ipykernel_3493536/4110811440.py:59: UserWarning: Glyph 38271 (\N{CJK UNIFIED IDEOGRAPH-957F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4110811440.py:59: UserWarning: Glyph 24230 (\N{CJK UNIFIED IDEOGRAPH-5EA6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4110811440.py:59: UserWarning: Glyph 39057 (\N{CJK UNIFIED IDEOGRAPH-9891}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4110811440.py:59: UserWarning: Glyph 27425 (\N{CJK UNIFIED IDEOGRAPH-6B21}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4110811440.py:59: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4110811440.py:59: UserWarning: Glyph 24067 (\N{CJK UNIFIED IDEOGRAPH-5E03}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/4110811440.py:59: UserWarning: Glyph 24179 (\N{CJK UN


统计图已保存到 outputs/figures/


/tmp/ipykernel_3493536/4110811440.py:60: UserWarning: Glyph 39057 (\N{CJK UNIFIED IDEOGRAPH-9891}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_random_policy_stats.png"),
/tmp/ipykernel_3493536/4110811440.py:60: UserWarning: Glyph 27425 (\N{CJK UNIFIED IDEOGRAPH-6B21}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_random_policy_stats.png"),
/tmp/ipykernel_3493536/4110811440.py:60: UserWarning: Glyph 38271 (\N{CJK UNIFIED IDEOGRAPH-957F}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_random_policy_stats.png"),
/tmp/ipykernel_3493536/4110811440.py:60: UserWarning: Glyph 24230 (\N{CJK UNIFIED IDEOGRAPH-5EA6}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_random_policy_stats.png"),
/tmp/ipykernel_3493536/4110811440.py:60: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_random_

## 7 — 强化学习的分类

强化学习算法可以根据不同维度分类。

### 7.1 Model-Based vs Model-Free

| 维度 | Model-Based | Model-Free |
|------|-------------|------------|
| **核心思路** | 学习/已知环境模型 $P(s'|s,a)$ 和 $R(s,a)$ | 不显式建模环境，直接从交互中学习 |
| **优势** | 样本效率高，可以"想象"规划 | 实现简单，不易受模型误差影响 |
| **劣势** | 模型误差会累积，复杂环境难以建模 | 样本效率低，需要大量交互 |
| **典型算法** | Dyna-Q, AlphaGo, MPC | Q-learning, DQN, PPO, SAC |

### 7.2 On-Policy vs Off-Policy

| 维度 | On-Policy | Off-Policy |
|------|-----------|------------|
| **核心思路** | 用当前策略 $\pi$ 产生的数据来学习 $\pi$ | 可以用其他策略（或旧策略）产生的数据学习 |
| **数据来源** | 必须使用当前策略生成的数据 | 可以使用 replay buffer 中的历史数据 |
| **样本效率** | 低（数据用完即弃） | 高（数据可以重复使用） |
| **稳定性** | 更稳定，不易发散 | 可能不稳定（分布偏移） |
| **典型算法** | SARSA, PPO, REINFORCE | Q-learning, DQN, SAC, DDPG |

### 7.3 其他分类维度

- **Value-based**：学习价值函数 $V(s)$ 或 $Q(s,a)$，隐式定义策略（如 Q-learning）
- **Policy-based**：直接学习策略 $\pi(a|s)$（如 REINFORCE, PPO）
- **Actor-Critic**：结合两者，Actor 学策略，Critic 学价值函数（如 A2C, SAC）


## 8 — 探索 vs 利用 (Exploration vs Exploitation)

### 核心问题

> **"我应该尝试新的动作来获取更多信息（探索），还是选择当前已知的最佳动作来最大化奖励（利用）？"**

这是 RL 中最根本的困境之一。

- **Exploitation（利用）**：选择当前认为最优的动作，最大化即时累积奖励
- **Exploration（探索）**：尝试未知的动作，获取信息以改进未来的决策

### 多臂老虎机 (Multi-Armed Bandit)

多臂老虎机是探索-利用困境的简化版本：
- 有 $K$ 台老虎机（臂），每台有未知的中奖概率 $\mu_k$
- 每次可以选择一个臂拉动，获得随机奖励
- 目标：在有限次拉动中最大化总奖励（或最小化 Regret）

下面我们用 Bandit 环境来对比不同策略的探索-利用平衡。


In [8]:
# ============================================================
# Cell 18: 创建多臂老虎机环境
# ============================================================

# 创建 10-臂 Bernoulli 老虎机
bandit = MultiArmedBandit(
    k=10,
    reward_type="bernoulli",
    seed=42,
)

print(f"臂的数量: {bandit.k}")
print(f"奖励类型: {bandit.reward_type}")
print(f"真实均值 (true_means): {bandit.true_means}")
print(f"最优臂索引: {bandit.optimal_action} (μ={bandit.true_means[bandit.optimal_action]:.4f})")
print()

# 测试拉动几次
for i in range(5):
    arm = i % bandit.k
    reward = bandit.pull(arm)
    print(f"拉臂 {arm}: 奖励 = {reward}")

print(f"\n总拉动次数: {bandit.total_pulls}")
print(f"各臂拉动次数: {bandit.action_counts}")
print(f"累积 regret: {bandit.regret:.4f}")


臂的数量: 10
奖励类型: bernoulli
真实均值 (true_means): [0.37454012 0.9507143  0.7319939  0.5986585  0.15601864 0.15599452
 0.05808361 0.8661761  0.601115   0.7080726 ]
最优臂索引: 1 (μ=0.9507)

拉臂 0: 奖励 = 1.0
拉臂 1: 奖励 = 0.0
拉臂 2: 奖励 = 0.0
拉臂 3: 奖励 = 1.0
拉臂 4: 奖励 = 0.0

总拉动次数: 5
各臂拉动次数: [1 1 1 1 1 0 0 0 0 0]
累积 regret: 2.7536


## 8.1 — 策略对比：随机 vs 贪心 vs ε-贪心

我们在 Bandit 上对比三种策略：

1. **Random（随机）**：每次都均匀随机选择一个臂
2. **Greedy（贪心）**：总是选择当前平均奖励最高的臂（不探索）
3. **ε-Greedy（ε-贪心）**：以概率 ε 随机探索，以概率 1-ε 贪心利用


In [9]:
# ============================================================
# Cell 20: Bandit 策略实现与对比
# ============================================================

def run_bandit_strategy(bandit, strategy, n_steps=1000, epsilon=0.1):
    '''在 bandit 上运行指定策略。

    Args:
        bandit: MultiArmedBandit 实例
        strategy: "random", "greedy", "epsilon_greedy"
        n_steps: 总步数
        epsilon: ε-贪心的探索概率

    Returns:
        rewards: 每步奖励数组
        actions: 每步选择的动作数组
    '''
    bandit.reset()
    rewards = np.zeros(n_steps)
    actions = np.zeros(n_steps, dtype=int)
    k = bandit.k

    # 初始化：每个臂至少拉一次（对于 greedy 和 epsilon-greedy）
    if strategy in ("greedy", "epsilon_greedy"):
        for a in range(k):
            rewards[a] = bandit.pull(a)
            actions[a] = a
        start_step = k
    else:
        start_step = 0

    for t in range(start_step, n_steps):
        if strategy == "random":
            action = bandit.rng.randint(0, k)

        elif strategy == "greedy":
            # 选择平均奖励最高的臂
            avg_rewards = bandit.action_rewards / np.maximum(bandit.action_counts, 1)
            action = int(np.argmax(avg_rewards))

        elif strategy == "epsilon_greedy":
            if bandit.rng.rand() < epsilon:
                action = bandit.rng.randint(0, k)  # 探索
            else:
                avg_rewards = bandit.action_rewards / np.maximum(bandit.action_counts, 1)
                action = int(np.argmax(avg_rewards))  # 利用

        rewards[t] = bandit.pull(action)
        actions[t] = action

    return rewards, actions


# 运行三种策略对比
n_steps = 2000
bandit_base = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)

results = {}
for name in ("random", "greedy", "epsilon_greedy"):
    # 每次使用独立的 bandit 实例确保公平对比
    b = MultiArmedBandit(k=10, reward_type="bernoulli", true_means=bandit_base.true_means.copy(), seed=42)
    rewards, actions = run_bandit_strategy(b, name, n_steps=n_steps, epsilon=0.1)
    results[name] = {
        "rewards": rewards,
        "actions": actions,
        "total_reward": rewards.sum(),
        "avg_reward": rewards.mean(),
        "best_arm_frac": (actions == b.optimal_action).mean(),
        "regret": b.regret,
    }
    print(f"=== {name} ===")
    print(f"  总奖励: {results[name]['total_reward']:.1f}")
    print(f"  平均奖励: {results[name]['avg_reward']:.4f}")
    print(f"  最优臂选择率: {results[name]['best_arm_frac']*100:.1f}%")
    print(f"  累积 Regret: {results[name]['regret']:.2f}")
    print()


=== random ===
  总奖励: 1036.0
  平均奖励: 0.5180
  最优臂选择率: 10.2%
  累积 Regret: 865.43

=== greedy ===
  总奖励: 752.0
  平均奖励: 0.3760
  最优臂选择率: 0.1%
  累积 Regret: 1149.43

=== epsilon_greedy ===
  总奖励: 1816.0
  平均奖励: 0.9080
  最优臂选择率: 91.0%
  累积 Regret: 85.43



In [10]:
# ============================================================
# Cell 21: Bandit 实验结果可视化
# ============================================================

def plot_bandit_comparison(results, n_steps, save_path=None):
    '''绘制 Bandit 策略对比图。'''
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 颜色与标签
    colors = {"random": "#E74C3C", "greedy": "#3498DB", "epsilon_greedy": "#2ECC71"}
    labels = {"random": "随机 (Random)", "greedy": "贪心 (Greedy)", "epsilon_greedy": "ε-贪心 (ε=0.1)"}

    # 1. 累积奖励曲线
    ax1 = axes[0, 0]
    for name, data in results.items():
        cumulative = np.cumsum(data["rewards"])
        ax1.plot(cumulative, color=colors[name], label=labels[name], linewidth=2)
    ax1.set_xlabel("步数")
    ax1.set_ylabel("累积奖励")
    ax1.set_title("累积奖励对比")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # 2. 滑动平均奖励
    ax2 = axes[0, 1]
    window = 100
    for name, data in results.items():
        avg_reward = np.convolve(data["rewards"], np.ones(window)/window, mode="valid")
        ax2.plot(avg_reward, color=colors[name], label=labels[name], linewidth=1.5)
    ax2.axhline(y=bandit_base.true_means.max(), color="gray", linestyle="--",
                label=f"最优臂均值 ({bandit_base.true_means.max():.3f})")
    ax2.set_xlabel("步数")
    ax2.set_ylabel(f"滑动平均奖励 (窗口={window})")
    ax2.set_title("奖励滑动平均")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # 3. 最优臂选择率
    ax3 = axes[1, 0]
    for name, data in results.items():
        actions = data["actions"]
        best_frac = np.array([
            (actions[:t+1] == bandit_base.optimal_action).mean()
            for t in range(n_steps)
        ])
        ax3.plot(best_frac, color=colors[name], label=labels[name], linewidth=1.5)
    ax3.set_xlabel("步数")
    ax3.set_ylabel("最优臂选择率")
    ax3.set_title("最优臂选择率变化")
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)

    # 4. 总奖励条形图
    ax4 = axes[1, 1]
    names = list(results.keys())
    total_rewards = [results[n]["total_reward"] for n in names]
    bar_colors = [colors[n] for n in names]
    bars = ax4.bar(range(len(names)), total_rewards, color=bar_colors, width=0.5, edgecolor="white")
    for bar, val in zip(bars, total_rewards):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f"{val:.0f}", ha="center", fontsize=11, fontweight="bold")
    ax4.set_xticks(range(len(names)))
    ax4.set_xticklabels([labels[n] for n in names], fontsize=10)
    ax4.set_ylabel("总奖励")
    ax4.set_title(f"总奖励对比 ({n_steps} 步)")
    ax4.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=120)
        print(f"图片已保存: {save_path}")
    plt.close(fig)


# 使用更精细的 long-run 结果绘图
n_steps_long = 2000
bandit_base2 = MultiArmedBandit(k=10, reward_type="bernoulli", seed=123)
long_results = {}
for name in ("random", "greedy", "epsilon_greedy"):
    b = MultiArmedBandit(k=10, reward_type="bernoulli",
                         true_means=bandit_base2.true_means.copy(), seed=123)
    rewards, actions = run_bandit_strategy(b, name, n_steps=n_steps_long, epsilon=0.1)
    long_results[name] = {"rewards": rewards, "actions": actions,
                          "total_reward": rewards.sum()}

plot_bandit_comparison(long_results, n_steps_long,
                       save_path=os.path.join(FIGURES_DIR, "01_bandit_comparison.png"))

print("\n=== Bandit 对比总结 ===")
print(f"{'策略':<20} {'总奖励':<10} {'最优臂%':<12} {'Regret':<10}")
print("-" * 52)
for name in ("random", "greedy", "epsilon_greedy"):
    b = MultiArmedBandit(k=10, reward_type="bernoulli",
                         true_means=bandit_base2.true_means.copy(), seed=123)
    r, a = run_bandit_strategy(b, name, n_steps=n_steps_long, epsilon=0.1)
    print(f"{name:<20} {r.sum():<10.1f} {(a==b.optimal_action).mean()*100:<11.1f}% {b.regret:<10.2f}")


/tmp/ipykernel_3493536/3932297377.py:68: UserWarning: Glyph 27493 (\N{CJK UNIFIED IDEOGRAPH-6B65}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/3932297377.py:68: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/3932297377.py:68: UserWarning: Glyph 32047 (\N{CJK UNIFIED IDEOGRAPH-7D2F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/3932297377.py:68: UserWarning: Glyph 31215 (\N{CJK UNIFIED IDEOGRAPH-79EF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/3932297377.py:68: UserWarning: Glyph 22870 (\N{CJK UNIFIED IDEOGRAPH-5956}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/3932297377.py:68: UserWarning: Glyph 21169 (\N{CJK UNIFIED IDEOGRAPH-52B1}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3493536/3932297377.py:68: UserWarning: Glyph 23545 (\N{CJK UN

/tmp/ipykernel_3493536/3932297377.py:70: UserWarning: Glyph 32047 (\N{CJK UNIFIED IDEOGRAPH-7D2F}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3932297377.py:70: UserWarning: Glyph 31215 (\N{CJK UNIFIED IDEOGRAPH-79EF}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3932297377.py:70: UserWarning: Glyph 22870 (\N{CJK UNIFIED IDEOGRAPH-5956}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3932297377.py:70: UserWarning: Glyph 21169 (\N{CJK UNIFIED IDEOGRAPH-52B1}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3932297377.py:70: UserWarning: Glyph 23545 (\N{CJK UNIFIED IDEOGRAPH-5BF9}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)
/tmp/ipykernel_3493536/3932297377.py:70: UserWarning: Glyph 

图片已保存: /workspace/data/vggt-omega/rl/outputs/figures/01_bandit_comparison.png

=== Bandit 对比总结 ===
策略                   总奖励        最优臂%         Regret    
----------------------------------------------------
random               1105.0     11.2       % 856.53    
greedy               1370.0     0.1        % 591.53    
epsilon_greedy       1754.0     70.2       % 207.53    


/tmp/ipykernel_3493536/3932297377.py:70: UserWarning: Glyph 24635 (\N{CJK UNIFIED IDEOGRAPH-603B}) missing from font(s) DejaVu Sans.
  fig.savefig(save_path, bbox_inches="tight", dpi=120)


## 8.2 — ε 参数对 ε-贪心策略的影响

ε 控制探索和利用的平衡：
- ε = 0：完全贪心（不探索，可能陷入次优）
- ε = 1：完全随机（不利用学到的信息）
- ε = 0.1：90% 利用 + 10% 探索（常用的折中）

下面我们扫描不同的 ε 值，观察它们对总奖励的影响。


In [11]:
# ============================================================
# Cell 23: ε 参数扫描
# ============================================================

def epsilon_sweep(bandit_template, n_steps=1000, epsilons=None, n_runs=20):
    '''对不同 ε 值运行多次实验，统计平均表现。'''
    if epsilons is None:
        epsilons = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

    results = {}
    for eps in epsilons:
        final_rewards = []
        for run in range(n_runs):
            b = MultiArmedBandit(
                k=bandit_template.k,
                reward_type=bandit_template.reward_type,
                true_means=bandit_template.true_means.copy(),
                seed=100 + run,
            )
            rewards, actions = run_bandit_strategy(b, "epsilon_greedy",
                                                    n_steps=n_steps, epsilon=eps)
            final_rewards.append(rewards.sum())

        results[eps] = {
            "mean": np.mean(final_rewards),
            "std": np.std(final_rewards),
            "se": np.std(final_rewards) / np.sqrt(n_runs),
        }
        print(f"  ε = {eps:.2f}: 平均总奖励 = {results[eps]['mean']:.1f} ± {results[eps]['se']:.1f}")

    return results


print("ε 参数扫描 (n_steps=1000, n_runs=20):")
bandit_template = MultiArmedBandit(k=10, reward_type="bernoulli", seed=42)
eps_results = epsilon_sweep(bandit_template, n_steps=1000, n_runs=20)

# 绘图
fig, ax = plt.subplots(figsize=(9, 5))
epsilons = sorted(eps_results.keys())
means = [eps_results[e]["mean"] for e in epsilons]
ses = [eps_results[e]["se"] for e in epsilons]

ax.errorbar(epsilons, means, yerr=ses, fmt="o-", color="#2ECC71",
            linewidth=2, markersize=8, capsize=5, capthick=2)
ax.axhline(y=np.mean(long_results["random"]["total_reward"]),
           color="#E74C3C", linestyle="--",
           label=f"随机策略参考 ({np.mean(long_results['random']['total_reward']):.0f})")
ax.set_xlabel("ε (探索概率)")
ax.set_ylabel(f"总奖励 (1000 步, 20-run 平均)")
ax.set_title("ε-贪心策略：ε 对总奖励的影响")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.savefig(os.path.join(FIGURES_DIR, "01_epsilon_sweep.png"),
            bbox_inches="tight", dpi=120)
print(f"\n图片已保存到 outputs/figures/")
plt.close(fig)


ε 参数扫描 (n_steps=1000, n_runs=20):
  ε = 0.00: 平均总奖励 = 922.7 ± 7.3


  ε = 0.01: 平均总奖励 = 919.8 ± 7.4
  ε = 0.05: 平均总奖励 = 903.1 ± 8.3


  ε = 0.10: 平均总奖励 = 885.6 ± 5.6
  ε = 0.20: 平均总奖励 = 848.0 ± 5.3


  ε = 0.50: 平均总奖励 = 726.6 ± 3.0
  ε = 1.00: 平均总奖励 = 515.6 ± 3.7



图片已保存到 outputs/figures/


/tmp/ipykernel_3493536/3891209119.py:55: UserWarning: Glyph 24635 (\N{CJK UNIFIED IDEOGRAPH-603B}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_epsilon_sweep.png"),
/tmp/ipykernel_3493536/3891209119.py:55: UserWarning: Glyph 22870 (\N{CJK UNIFIED IDEOGRAPH-5956}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_epsilon_sweep.png"),
/tmp/ipykernel_3493536/3891209119.py:55: UserWarning: Glyph 21169 (\N{CJK UNIFIED IDEOGRAPH-52B1}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_epsilon_sweep.png"),
/tmp/ipykernel_3493536/3891209119.py:55: UserWarning: Glyph 27493 (\N{CJK UNIFIED IDEOGRAPH-6B65}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_epsilon_sweep.png"),
/tmp/ipykernel_3493536/3891209119.py:55: UserWarning: Glyph 24179 (\N{CJK UNIFIED IDEOGRAPH-5E73}) missing from font(s) DejaVu Sans.
  fig.savefig(os.path.join(FIGURES_DIR, "01_epsilon_sweep.png"),
/tmp/ipyke

## 9 — RL vs 监督学习 vs 规划

| 维度 | 强化学习 (RL) | 监督学习 (SL) | 规划 (Planning) |
|------|--------------|---------------|-----------------|
| **数据来源** | 智能体主动交互产生 | 给定的标记数据集 | 给定的环境模型 |
| **反馈信号** | 延迟的奖励 $r_t$（稀疏，有噪声） | 即时监督标签 $y$（精确） | 无（模型已知） |
| **数据分布** | 非独立同分布（动作影响未来数据） | 独立同分布假设 | 无数据生成问题 |
| **目标** | 最大化累积奖励 $\max \mathbb{E}[G_0]$ | 最小化泛化误差 $\min \mathcal{L}(y, \hat{y})$ | 找到最优路径/动作序列 |
| **探索需求** | 必须（否则无法发现好策略） | 不需要 | 不需要 |
| **信用分配** | 需要解决（哪个动作导致了好结果？） | 不需要（每个样本独立） | 不需要（已知模型） |
| **环境模型** | 可选（model-based 需要） | 不需要 | 必须已知 |
| **典型应用** | 游戏、机器人、推荐系统 | 图像分类、NLP、预测 | 路径规划、调度 |

### 关键区别说明

**RL vs 监督学习：**
- 监督学习假设数据是独立同分布的，RL 中当前决策影响未来数据分布
- 监督学习有即时正确的标签，RL 只有延迟且可能有噪声的奖励
- RL 需要平衡探索和利用，监督学习不需要

**RL vs 规划：**
- 规划假设环境模型完全已知（转移概率 $P$ 和奖励函数 $R$）
- RL 在不了解环境模型的情况下也能学习
- Model-based RL 可看作两者的结合：先学模型，再规划


## 10 — Tabular vs Function Approximation

### Tabular Method（表格方法）

对于状态空间和动作空间都较小的问题，可以直接用表格存储价值函数：

$$ Q(s, a) \in \mathbb{R}^{|\mathcal{S}| \times |\mathcal{A}|} $$

- **优点**：精确，每种状态-动作对独立更新，无泛化偏差
- **缺点**：无法扩展到大规模状态空间（如围棋 $10^{170}$ 状态）
- **典型算法**：Q-learning、SARSA、Dynamic Programming

### Function Approximation（函数近似）

对于大规模或连续状态空间，使用参数化函数来近似价值函数或策略：

$$ Q_\theta(s, a) \approx Q^*(s, a) \quad \text{或} \quad \pi_\theta(a|s) \approx \pi^*(a|s) $$

- **优点**：泛化到未见过状态，处理大规模/连续空间
- **缺点**：可能存在泛化偏差，训练不稳定
- **典型方法**：
  - 线性函数近似：$Q_\theta(s, a) = \theta^\top \phi(s, a)$
  - 神经网络：Deep Q-Networks (DQN)、Policy Networks

**过渡：** 本课程后续将先学习 Tabular 方法（第 02-05 课），再逐步过渡到深度 RL（第 06 课之后）。


## 11 — 本课总结

### 核心要点回顾

1. **强化学习** 是智能体通过与环境交互、最大化累积奖励的试错学习范式
2. **交互循环**：Agent $\xrightarrow{a_t}$ Environment $\xrightarrow{s_{t+1}, r_t}$ Agent
3. **核心概念**：状态 $s_t$、动作 $a_t$、奖励 $r_t$、回报 $G_t = \sum \gamma^k r_{t+k+1}$、策略 $\pi(a|s)$
4. **回报** 折扣累积未来奖励，折扣因子 $\gamma$ 控制远见程度
5. **RL 分类**：
   - Model-Based vs Model-Free（是否学习环境模型）
   - On-Policy vs Off-Policy（用当前策略还是历史数据学习）
6. **探索-利用困境**：Bandit 实验显示 ε-贪心通常优于纯随机或纯贪心
7. **RL vs 监督学习 vs 规划**：RL 需要处理延迟反馈、非独立同分布数据、探索需求
8. **Tabular vs 函数近似**：小规模用表格，大规模用函数近似（神经网络）

### 下一步

在下一课中，我们将深入**马尔可夫决策过程 (MDP)** 的形式化定义，这是几乎所有 RL 问题的数学基础。


## 12 — 面试问题（点击展开查看答案）

<details>
<summary><b>问题 1：</b> 请解释强化学习中的"信用分配问题"(Credit Assignment Problem)。</summary>

<br>
信用分配问题是 RL 中的核心挑战之一：当智能体在 episode 结束时获得奖励（正或负）时，它如何确定是之前的哪个动作导致了这个结果？

例如，在 GridWorld 中，智能体走了 20 步到达目标并获得 +10 奖励。是第 1 步的动作、第 15 步的动作、还是所有动作共同作用的结果？奖励在时间上延迟，智能体必须学会将"信用"分配给真正重要的动作。

折扣回报 $G_t = \sum \gamma^k r_{t+k+1}$ 是一种解决方法：它更强调近期动作的影响，但远期动作仍然通过折扣因子 $\gamma$ 得到部分信用。更复杂的方法包括 Eligibility Traces 和 Advantage 估计。
</details>

<br>

<details>
<summary><b>问题 2：</b> 折扣因子 γ 的作用是什么？γ=0 和 γ=1 分别代表什么？</summary>

<br>
折扣因子 $\gamma \in [0, 1]$ 控制智能体对远期奖励的重视程度：

- **γ = 0**：仅关心即时奖励（短视），$G_t = r_{t+1}$。智能体完全不在乎未来。
- **γ = 1**：所有未来奖励权重相同（远视），$G_t = \sum_{k=0}^{\infty} r_{t+k+1}$。可能发散（无限 horizon）。
- **中间值**（0.9-0.99）：常用的折中，兼顾近期和远期。

γ 还提供了数学上的便利：保证无限 horizon 下的回报收敛（当 γ < 1 且奖励有界时）。
</details>

<br>

<details>
<summary><b>问题 3：</b> On-Policy 和 Off-Policy 学习有什么区别？各举一个算法例子。</summary>

<br>
**On-Policy**：使用当前策略 $\pi$ 产生的交互数据来学习/改进同一个策略 $\pi$。数据用完即弃，不能重复使用。

- 例子：SARSA, PPO, REINFORCE
- 优点：更稳定，保证收敛
- 缺点：样本效率低

**Off-Policy**：可以使用任意行为策略（包括旧版本的策略）产生的数据来学习目标策略。数据可以存放在 Replay Buffer 中重复使用。

- 例子：Q-learning, DQN, SAC
- 优点：样本效率高，可以使用历史数据
- 缺点：可能存在分布偏移问题，需要重要性采样校正

**类比**：On-Policy 像一个厨师只吃自己做的菜来改进；Off-Policy 像一个厨师可以吃任何菜（包括自己以前做的）来学习。
</details>

<br>

<details>
<summary><b>问题 4：</b> 什么是探索-利用困境？ε-贪心策略如何工作？</summary>

<br>
探索-利用困境 (Exploration-Exploitation Dilemma) 是指：

- **利用**：选择当前已知的最佳选项，最大化即时奖励
- **探索**：尝试未知的选项，获取信息以改进未来决策

两者冲突：多探索则少即时奖励，多利用则可能错过更好的选择。

**ε-贪心策略 (ε-Greedy)：**
- 以概率 $(1-\varepsilon)$ 选择当前估计最优的动作（利用）
- 以概率 $\varepsilon$ 随机选择一个动作（探索）
- $\varepsilon$ 通常取 0.05-0.2
- 简单的改进：$\varepsilon$ 随时间衰减（早期多探索，后期多利用）
</details>

<br>

<details>
<summary><b>问题 5：</b> State 和 Observation 有什么区别？</summary>

<br>
**State (状态) $s_t$**：环境的完整、内部描述，包含做出最优决策所需的全部信息。

**Observation (观测) $o_t$**：智能体实际接收到的信息，可能是状态的子集或噪声版本。

- 在完全可观测环境 (Fully Observable) 中：$o_t = s_t$（如围棋、GridWorld）
- 在部分可观测环境 (Partially Observable, POMDP) 中：$o_t \neq s_t$（如扑克——看不到对手的手牌、自动驾驶——雷达视野有限）

POMDP 中智能体通常需要维护对状态的后验信念或使用 RNN 记忆历史观测。
</details>

<br>

<details>
<summary><b>问题 6：</b> 请解释回报 (Return) 和奖励 (Reward) 的区别。</summary>

<br>
**奖励 (Reward) $r_t$**：即时信号，表示在时刻 $t$ 获得的好坏程度。

**回报 (Return) $G_t$**：从时刻 $t$ 开始所有未来折扣奖励的累积和。

$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1} = r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots$$

区别：
- 奖励是"即时反馈"，回报是"长期目标"
- 智能体的目标不是最大化即时奖励，而是最大化回报
- 一个动作可能产生负面即时奖励但有高回报（如围棋中弃子以获得最终胜利）
</details>


## 13 — 练习

### 练习 1：修改 GridWorld 配置

创建一个新的 GridWorld，要求：
- 6×6 网格
- 起点 (0,0)，终点 (5,5)
- 至少 4 个障碍物，形成一条"迷宫"路径
- 每步奖励 -0.5，目标奖励 +20
- 运行一个随机策略 episode，打印轨迹和折扣回报

### 练习 2：比较不同 ε 值

在 10-臂 Bernoulli Bandit 上运行 ε-贪心策略，对比 ε ∈ {0, 0.01, 0.1, 0.5}，各运行 5000 步，重复 10 次：
- 绘制不同 ε 的累积奖励曲线（带误差带）
- 计算每个 ε 的最终平均奖励和 Regret
- 哪个 ε 表现最好？为什么？

### 练习 3：实现乐观初始化 (Optimistic Initialization)

在 Bandit 策略中，将每个臂的初始估计值设为 5（而非 0），然后运行贪心策略：
- 乐观初始化为什么不探索也能工作？
- 对比乐观初始化贪心 vs 标准 ε-贪心 (ε=0.1)

### 练习 4：UCB 策略

实现 UCB (Upper Confidence Bound) 策略：

$$a_t = \arg\max_a \left[ \hat{\mu}_a + c \sqrt{\frac{\ln t}{N_a}} \right]$$

其中 $\hat{\mu}_a$ 是臂 $a$ 的平均奖励，$N_a$ 是臂 $a$ 被拉的次数，$c$ 是探索系数。

对比 UCB 与 ε-贪心在 Bandit 上的表现。

### 练习 5：思考题

1. 为什么 RL 中的数据不是独立同分布的？这对学习算法有什么影响？
2. 在什么场景下 Model-Based RL 比 Model-Free 更好？什么场景下相反？
3. 设计一个现实生活中的 RL 问题（非游戏类），明确其中的 State、Action、Reward 是什么。


## 14 — 扩展阅读

### 经典教材

| 书名 | 作者 | 备注 |
|------|------|------|
| **Reinforcement Learning: An Introduction (2nd ed.)** | Sutton & Barto | RL 圣经，必读经典 |
| **Algorithms for Reinforcement Learning** | Csaba Szepesvari | 简洁的理论介绍 |
| **Deep Reinforcement Learning Hands-On** | Maxim Lapan | 实践导向，含代码实现 |

### 重要论文

- **Playing Atari with Deep Reinforcement Learning** (Mnih et al., 2013) — DQN 的诞生
- **Human-level control through deep reinforcement learning** (Mnih et al., 2015) — Nature DQN
- **Mastering the Game of Go with Deep Neural Networks and Tree Search** (Silver et al., 2016) — AlphaGo
- **Proximal Policy Optimization Algorithms** (Schulman et al., 2017) — PPO
- **Soft Actor-Critic: Off-Policy Maximum Entropy Deep RL** (Haarnoja et al., 2018) — SAC

### 在线资源

- [Spinning Up in Deep RL (OpenAI)](https://spinningup.openai.com/) — 实践导向的深度 RL 教程
- [RL Course by David Silver](https://www.davidsilver.uk/teaching/) — 经典 RL 课程视频
- [Deep RL Course (UC Berkeley)](https://rail.eecs.berkeley.edu/deeprlcourse/) — CS 285
- [OpenAI Gym](https://www.gymlibrary.dev/) — 标准 RL 环境库

---

*本 Notebook 使用 nbformat 自动生成。*
